Библиотеки:

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import MACCSkeys
from rdkit.Avalon import pyAvalonTools
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [ ]:
data = pd.read_csv("ToxicDataset.csv")
data.head()

,TAID,Pubchem CID,IUPAC Name,SMILES,Canonical SMILES,InChIKey,mouse_intraperitoneal_LD50
0,TOX-145,785,"benzene-1,4-diol",Oc1ccc(O)cc1,Oc1ccc(O)cc1,QIGBRXMKCJKVMJ-UHFFFAOYSA-N,3.041835
1,TOX-245,5453,tris(aziridin-1-yl)-sulfanylidene-lambda5-phos...,S=P(N1CC1)(N1CC1)N1CC1,S=P(N1CC1)(N1CC1)N1CC1,FOCVUCIESVLUNU-UHFFFAOYSA-N,4.235584
2,TOX-1273,727,"1,2,3,4,5,6-hexachlorocyclohexane",ClC1C(Cl)C(Cl)C(Cl)C(Cl)C1Cl,ClC1C(Cl)C(Cl)C(Cl)C(Cl)C1Cl,JLYXXMFPNIAWKQ-UHFFFAOYSA-N,3.366732
3,TOX-1279,4091,"3-(diaminomethylidene)-1,1-dimethylguanidine",CN(C)C(=N)N=C(N)N,CN(C)C(=N)N=C(N)N,XZWYZXLIPXDOLR-UHFFFAOYSA-N,2.641604
4,TOX-1282,10364,2-methyl-5-propan-2-ylphenol,Cc1ccc(C(C)C)cc1O,Cc1ccc(C(C)C)cc1O,RECUKUPTGUEGMW-UHFFFAOYSA-N,3.311627


In [ ]:
# Считаем количество пропусков и дубликатов.
nans = []
for column in data.columns:
    print(column)
    print(data[column].isnull().sum())
    nans.append(data[column].isnull().sum())
print(f'Общее число пропусков: {sum(nans)}')
print(f'Общее число дубликатов: {data.duplicated().sum()}')

TAID
0
Pubchem CID
0
IUPAC Name
1919
SMILES
0
Canonical SMILES
0
InChIKey
38
mouse_intraperitoneal_LD50
0
Общее число пропусков: 1957
Общее число дубликатов: 0


Общее количество строк с пропусками значительно меньше всего объема датасета, поэтому их можно спокойно удалить. Дубликаты отсутствуют.

In [ ]:
# Удаляем пропуски
data = data.dropna()
data.reset_index(drop=True, inplace=True)

In [ ]:
# Среднее значение длины SMILES
def smiles_len(smiles):
    return len(smiles)
data['SMILES_len'] = data['SMILES'].apply(smiles_len)
print(f'Средняя длина SMILES: {data["SMILES_len"].mean()}')

# Среднее количество атомов в молекуле без водородов
def num_of_atoms(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return mol.GetNumAtoms()
data['AtomCount'] = data['SMILES'].apply(num_of_atoms)
print(f'Среднее количество атомов в молекуле без водородов: {data["AtomCount"].mean()}')
data.head()

# Среднее количество атомов в молекуле с водородами
def num_of_atoms_with_h(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mol_with_h = Chem.AddHs(mol)
    return mol_with_h.GetNumAtoms()
data['HAtomCount'] = data['SMILES'].apply(num_of_atoms_with_h)
print(f'Среднее количество атомов в молекуле c водородами: {data["HAtomCount"].mean()}')


Средняя длина SMILES: 35.941515206046425
Среднее количество атомов в молекуле без водородов: 21.505848479395357
Среднее количество атомов в молекуле c водородами: 41.770679623297944


,TAID,Pubchem CID,IUPAC Name,SMILES,Canonical SMILES,InChIKey,mouse_intraperitoneal_LD50,SMILES_len,AtomCount,HAtomCount
0,TOX-145,785,"benzene-1,4-diol",Oc1ccc(O)cc1,Oc1ccc(O)cc1,QIGBRXMKCJKVMJ-UHFFFAOYSA-N,3.041835,12,8,14
1,TOX-245,5453,tris(aziridin-1-yl)-sulfanylidene-lambda5-phos...,S=P(N1CC1)(N1CC1)N1CC1,S=P(N1CC1)(N1CC1)N1CC1,FOCVUCIESVLUNU-UHFFFAOYSA-N,4.235584,22,11,23
2,TOX-1273,727,"1,2,3,4,5,6-hexachlorocyclohexane",ClC1C(Cl)C(Cl)C(Cl)C(Cl)C1Cl,ClC1C(Cl)C(Cl)C(Cl)C(Cl)C1Cl,JLYXXMFPNIAWKQ-UHFFFAOYSA-N,3.366732,28,12,18
3,TOX-1279,4091,"3-(diaminomethylidene)-1,1-dimethylguanidine",CN(C)C(=N)N=C(N)N,CN(C)C(=N)N=C(N)N,XZWYZXLIPXDOLR-UHFFFAOYSA-N,2.641604,17,9,20
4,TOX-1282,10364,2-methyl-5-propan-2-ylphenol,Cc1ccc(C(C)C)cc1O,Cc1ccc(C(C)C)cc1O,RECUKUPTGUEGMW-UHFFFAOYSA-N,3.311627,17,11,25


Средняя длина SMILES составляет около 36 символов, а среднее количество атомов в молекуле - 42. Из этого можно сделать вывод, что это довольно большие органические молекулы.

In [ ]:
# Молекулярные дескрипторы
def get_descriptor(smiles, descriptor_func):
    mol = Chem.MolFromSmiles(smiles)
    return round(descriptor_func(mol), 2)

descriptors = {
    'MolWt': Descriptors.ExactMolWt,
    'Radical_e': Descriptors.NumRadicalElectrons,
    'Valence_e': Descriptors.NumValenceElectrons,
    'H_Donors': Descriptors.NumHDonors,
    'H_Acceptors': Descriptors.NumHAcceptors,
    'Num_Rings': Descriptors.RingCount,
    'TPSA': Descriptors.TPSA,
    'LogP': Descriptors.MolLogP
}

for desc_name, desc_func in descriptors.items():
    data[desc_name] = data['Canonical SMILES'].apply(lambda smiles: get_descriptor(smiles, desc_func))


In [ ]:
# Фингерпринты MACCS_keys
def generate_MACCS(data):
    MACCS_keys = []
    mols = [Chem.MolFromSmiles(x) for x in data]
    for mol in tqdm(mols):
        maccs = list(MACCSkeys.GenMACCSKeys(mol).ToBitString())
        MACCS_keys.append(maccs)
    return np.array(MACCS_keys)

MACCS_keys = generate_MACCS(data['Canonical SMILES'])
maccs_data = pd.DataFrame(MACCS_keys)
maccs_names = [f'MACCS_{i}' for i in range(1, 168)]
maccs_data.columns = maccs_names

# Фингерпринты Avalon
def generate_Avalon(data):
    Avalon_fpts = []
    mols = [Chem.MolFromSmiles(x) for x in data]
    for mol in tqdm(mols):
        avalon = pyAvalonTools.GetAvalonFP(mol, nBits=512)
        Avalon_fpts.append(avalon)
    return np.array(Avalon_fpts)

Avalon_fpts = generate_Avalon(data['Canonical SMILES'])
avalon_data = pd.DataFrame(Avalon_fpts)
avalon_names = [f'A_{i}' for i in range(1, 513)]
avalon_data.columns = avalon_names

new_data = pd.concat([data, maccs_data, avalon_data], axis=1)

100%|██████████| 33342/33342 [00:22<00:00, 1464.26it/s]


In [ ]:
# Создаю новый датасет с дескрипторами
columns_to_exclude = ['Pubchem CID', 'IUPAC Name', 'SMILES', 'InChIKey', 'SMILES_len', 'AtomCount', 'HAtomCount']
toxins_data = new_data.drop(columns=columns_to_exclude)
toxins_data.to_csv('toxin_data.csv', index=False)

In [ ]:
# Создание модели и ее обучение
for column in maccs_names:
    toxins_data[column] = toxins_data[column].astype('category')

X = toxins_data.drop(columns=['mouse_intraperitoneal_LD50',
                              'TAID', 'Canonical SMILES'])
y = toxins_data['mouse_intraperitoneal_LD50']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = CatBoostRegressor(iterations=1000, learning_rate=0.1, depth=6, verbose=100)
model.fit(X_train, y_train, cat_features=maccs_names)
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
print(f"Коэффициент детерминации R^2: {r2}")

0:	learn: 0.6823880	total: 163ms	remaining: 2m 42s
100:	learn: 0.5043862	total: 3.2s	remaining: 28.5s
200:	learn: 0.4552857	total: 6.24s	remaining: 24.8s
300:	learn: 0.4231037	total: 9.26s	remaining: 21.5s
400:	learn: 0.4001998	total: 12.3s	remaining: 18.4s
500:	learn: 0.3812995	total: 15.4s	remaining: 15.3s
600:	learn: 0.3647917	total: 18.4s	remaining: 12.2s
700:	learn: 0.3509313	total: 21.4s	remaining: 9.14s
800:	learn: 0.3379963	total: 24.5s	remaining: 6.09s
900:	learn: 0.3270482	total: 27.5s	remaining: 3.03s
999:	learn: 0.3169120	total: 30.6s	remaining: 0us
Коэффициент детерминации R^2: 0.6154047698574832


Вывод:

У меня получилось создать и обучить модель машинного обучения для определения параметра токсичности молекул. В качестве дескрипторов молекул использовались модуль rdkit.Chem.Descriptors, фингерпринты MACCS_keys и Avalon. Изначально модель была обучена только с фингерпринтами MACCS_keys и молекулярнами дескрипторами, точность модели составляла 0.57. Затем я добавил фингерпринты Avalon, после этого коэффициент детерминациии составил 0.62, что уже можно считать довольно неплохим результатом. Очевидно, что с добавлением предикторов модель ставновится точнее, однако стоит обратить внимание на то, не переобучается ли модель. Кроме того, для улушчения качества модели можно воспользоваться другими моделями и подбором категориальных признаков.